# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growth prediction

The FlyRank paper reports that its growth-prediction model, trained on about 96.6K clearly growing or declining pages, achieved about 90% accuracy on unseen pages from represented brands and about 75% on completely unseen brands.

**Methodology question:** Does the model still perform well on pages close to the growth/decline cutoff, or is most of its accuracy coming from pages with very obvious large changes?

This matters because restricting evaluation to clear positive and negative cases can make a classification problem easier than the borderline cases where a real decision is hardest. A useful audit would stratify performance by distance from the growth/decline threshold and check whether discrimination remains useful near that boundary.

### Finding 2 — 30-day momentum

The paper reports that its 30-day momentum model predicts whether a page will improve by more than 10% in the following month, with strong reported performance on both unseen pages from represented brands and completely unseen brands.

**Methodology question:** Were repeated observations from the same page or brand kept together during validation so that closely related observations could not appear in both training and test sets?

This matters because related observations can share persistent page- or brand-level behaviour. Keeping each repeated entity on only one side of a split tests whether performance transfers to genuinely independent entities rather than benefiting from information shared across closely related observations.


In [1]:
# Section 1 is a methodological reading exercise; no paper re-analysis is claimed here.
paper_questions = {
    "growth_prediction": "Does performance remain strong near the growth/decline cutoff?",
    "momentum_validation": "Were repeated page/brand observations kept together during validation?",
}
paper_questions


{'growth_prediction': 'Does performance remain strong near the growth/decline cutoff?',
 'momentum_validation': 'Were repeated page/brand observations kept together during validation?'}

## 2. My model under an honest split (before/after)

Assignment 6 already used client-grouped cross-validation, so I did not replace an actually used random split and pretend it was my historical Week-5 design. Instead, I reconstruct a **counterfactual naive before** using shuffled page-level 5-fold CV and compare it with the **honest after** using 5-fold `GroupKFold` by client.

Everything except the split is frozen: the same 1,800-page Assignment-6 training population, the same five March-only features, the same April targets, the same selected Random Forest hyperparameters, and the same ranking blend (`gamma=2`, `lambda=1`, with the Assignment-6 severity scale). This isolates the effect of allowing versus preventing the same client from appearing in both fit and validation folds.

### Observed before/after

| Metric | Random page CV (before) | Client-grouped CV (after) |
|---|---:|---:|
| Classification ROC-AUC | 0.7247 | 0.6650 |
| Regression RMSE | 0.8186 | 0.8063 |
| Ranking Precision@50 | 0.9480 | 0.8720 |
| Mean train/validation client overlap | 15 | 0 |

The overall decline base rate in this 1,800-page development population is **0.7511**. Under random page CV, every validation fold contains pages from all 15 clients that are also represented in its fitting data. Under grouped CV, client overlap is exactly zero.

The measured classification ROC-AUC falls from 0.7247 to 0.6650 and Precision@50 falls from 0.9480 to 0.8720 when client overlap is removed. This is directional evidence that the random page split gives a more optimistic view of those two tasks when the intended question is generalisation to unseen clients. Regression does **not** follow that pattern: RMSE changes from 0.8186 to 0.8063, a small improvement under grouping, so I do not claim that random splitting inflated regression performance.

The grouped folds are also much more heterogeneous: their decline prevalence ranges from about 0.664 to 0.936, and the corresponding model metrics vary more across folds. I therefore treat the grouped estimates as the more appropriate decision-support evidence for unseen-client performance, while the random-split results are retained only as a comparison showing what changes when entity independence is not enforced.


In [2]:
# SECTION 2 — random page CV vs client-grouped CV using the frozen Assignment-6 models.
# The only intentional change is the split design.

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.model_selection import GroupKFold, KFold

# ---------- warehouse access ----------
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required to rebuild the exact Assignment-6 frame.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# ---------- rebuild the exact locked 2,520-page population ----------
march_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[
    matched_keys["april_usable_days"] >= 20
][["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)
client_tier_counts = (
    march_exposure.groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size().unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
eligible_clients = client_tier_counts[client_tier_counts.min(axis=1) >= 40].index.tolist()
balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(["client_hash_id", "exposure_tier"], observed=False, group_keys=False)
    .head(40).reset_index(drop=True)
)
balanced_keys = balanced_poc[["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("balanced_keys", balanced_keys)

march_features = con.sql(f"""
    WITH daily AS (
        SELECT f.client_hash_id, f.content_hash_id, f.report_date,
               DATE_DIFF('day', DATE '2026-03-01', f.report_date)::DOUBLE AS day_index,
               f.gsc_impressions::DOUBLE AS impressions,
               f.gsc_clicks::DOUBLE AS clicks,
               CASE WHEN f.gsc_avg_position >= 1
                    THEN f.gsc_avg_position::DOUBLE ELSE NULL END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
           MEDIAN(valid_position) AS median_position,
           REGR_SLOPE(valid_position, day_index)
               FILTER (WHERE valid_position IS NOT NULL) AS position_slope_per_day,
           QUANTILE_CONT(valid_position, 0.75) - QUANTILE_CONT(valid_position, 0.25)
               AS position_iqr
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

age_feature = con.sql(f"""
    SELECT d.client_hash_id, d.content_hash_id,
           DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')::DOUBLE
               AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k USING (client_hash_id, content_hash_id)
""").df()

FINAL_FEATURES = [
    "aggregate_ctr", "median_position", "position_slope_per_day",
    "position_iqr", "content_age_days"
]
feature_frame = (
    march_features.merge(age_feature, on=["client_hash_id", "content_hash_id"], how="inner")
    .sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
)

con.register("model_keys", feature_frame[["client_hash_id", "content_hash_id"]].drop_duplicates())
march_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN model_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()
april_target = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
           SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
               AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN model_keys AS k USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()
target_frame = march_target.merge(april_target, on=["client_hash_id", "content_hash_id"], how="inner")
target_frame["future_impression_change"] = (
    target_frame["april_avg_impressions_per_day"] - target_frame["march_avg_impressions_per_day"]
) / target_frame["march_avg_impressions_per_day"]
target_frame["future_decline"] = (target_frame["future_impression_change"] < 0).astype(int)

modeling_frame = (
    feature_frame.merge(
        target_frame[["client_hash_id", "content_hash_id", "future_impression_change", "future_decline"]],
        on=["client_hash_id", "content_hash_id"], how="inner"
    )
    .sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
)

# Reuse the exact Assignment-5/6 training-client population; the six external clients remain untouched.
output_dir = Path("../outputs")
with open(output_dir / "baseline_split_manifest.json", "r", encoding="utf-8") as fh:
    split_manifest = json.load(fh)
train_clients = set(split_manifest["train_clients"])
train_frame = (
    modeling_frame[modeling_frame["client_hash_id"].isin(train_clients)]
    .sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
)

assert len(modeling_frame) == 2520
assert modeling_frame["client_hash_id"].nunique() == 21
assert len(train_frame) == 1800
assert train_frame["client_hash_id"].nunique() == 15
assert train_frame[FINAL_FEATURES].notna().all().all()

X = train_frame[FINAL_FEATURES].copy()
y_cls = train_frame["future_decline"].astype(int).copy()
y_reg = train_frame["future_impression_change"].astype(float).copy()
groups = train_frame["client_hash_id"].copy()

# ---------- freeze the Assignment-6 selected models ----------
cls_model = RandomForestClassifier(
    n_estimators=400, random_state=42, n_jobs=-1,
    class_weight=None, max_depth=4, max_features="sqrt", min_samples_leaf=5,
)
reg_model = RandomForestRegressor(
    n_estimators=400, random_state=42, n_jobs=-1,
    max_depth=None, max_features="sqrt", min_samples_leaf=30,
)
RANK_GAMMA = 2.0
RANK_LAMBDA = 1.0
SEVERITY_SCALE = 0.47977155635777924

def evaluate_split(split_name, splits):
    rows = []
    for fold, (fit_idx, valid_idx) in enumerate(splits, start=1):
        X_fit, X_valid = X.iloc[fit_idx], X.iloc[valid_idx]
        yc_fit, yc_valid = y_cls.iloc[fit_idx], y_cls.iloc[valid_idx]
        yr_fit, yr_valid = y_reg.iloc[fit_idx], y_reg.iloc[valid_idx]

        c = clone(cls_model).fit(X_fit, yc_fit)
        r = clone(reg_model).fit(X_fit, yr_fit)
        p = c.predict_proba(X_valid)[:, 1]
        pred_r = r.predict(X_valid)

        auc = float(roc_auc_score(yc_valid, p))
        rmse = float(np.sqrt(mean_squared_error(yr_valid, pred_r)))

        severity = np.maximum(0.0, -pred_r)
        severity_norm = np.clip(severity / SEVERITY_SCALE, 0.0, 1.0)
        rank_score = np.power(np.clip(p, 1e-9, 1.0), RANK_GAMMA) * (
            1.0 + RANK_LAMBDA * severity_norm
        )
        rank_df = pd.DataFrame({"score": rank_score, "relevance": yc_valid.to_numpy()})
        rank_df = rank_df.sort_values("score", ascending=False)
        k = min(50, len(rank_df))
        p50 = float(rank_df.head(k)["relevance"].mean())

        fit_clients = set(groups.iloc[fit_idx])
        valid_clients = set(groups.iloc[valid_idx])
        rows.append({
            "split": split_name,
            "fold": fold,
            "validation_pages": int(len(valid_idx)),
            "validation_clients": int(len(valid_clients)),
            "client_overlap": int(len(fit_clients.intersection(valid_clients))),
            "decline_base_rate": float(yc_valid.mean()),
            "classification_roc_auc": auc,
            "regression_rmse": rmse,
            "ranking_precision_at_50": p50,
        })
    return pd.DataFrame(rows)

random_cv = KFold(n_splits=5, shuffle=True, random_state=42)
grouped_cv = GroupKFold(n_splits=5)

random_rows = evaluate_split("random_page_cv", list(random_cv.split(X, y_cls)))
grouped_rows = evaluate_split("grouped_client_cv", list(grouped_cv.split(X, y_cls, groups=groups)))
fold_results = pd.concat([random_rows, grouped_rows], ignore_index=True)

summary = (
    fold_results.groupby("split", as_index=False)
    .agg(
        mean_client_overlap=("client_overlap", "mean"),
        mean_decline_base_rate=("decline_base_rate", "mean"),
        classification_roc_auc=("classification_roc_auc", "mean"),
        classification_roc_auc_sd=("classification_roc_auc", "std"),
        regression_rmse=("regression_rmse", "mean"),
        regression_rmse_sd=("regression_rmse", "std"),
        ranking_precision_at_50=("ranking_precision_at_50", "mean"),
        ranking_precision_at_50_sd=("ranking_precision_at_50", "std"),
    )
)

# The grouped rerun should reproduce the frozen Assignment-6 CV results.
grouped_summary = summary[summary["split"] == "grouped_client_cv"].iloc[0]
assert np.isclose(grouped_summary["classification_roc_auc"], 0.6650253727996942)
assert np.isclose(grouped_summary["regression_rmse"], 0.806274586196414)
assert np.isclose(grouped_summary["ranking_precision_at_50"], 0.8720000000000001)
assert (grouped_rows["client_overlap"] == 0).all()
assert (random_rows["client_overlap"] > 0).all()

receipt = {
    "comparison_role": "counterfactual naive random-page CV vs honest client-grouped CV",
    "population": {"pages": 1800, "clients": 15},
    "features": FINAL_FEATURES,
    "decline_base_rate": float(y_cls.mean()),
    "frozen_models": {
        "classifier": "RandomForestClassifier(max_depth=4, max_features='sqrt', min_samples_leaf=5, n_estimators=400)",
        "regressor": "RandomForestRegressor(max_features='sqrt', min_samples_leaf=30, n_estimators=400)",
        "ranking": {"gamma": RANK_GAMMA, "lambda": RANK_LAMBDA, "severity_scale": SEVERITY_SCALE},
    },
    "folds": fold_results.to_dict(orient="records"),
    "summary": summary.to_dict(orient="records"),
}
receipt_path = output_dir / "assignment7_split_audit.json"
with open(receipt_path, "w", encoding="utf-8") as fh:
    json.dump(receipt, fh, indent=2)

print("ASSIGNMENT 7 — HONEST SPLIT AUDIT")
print("Pages:", len(train_frame))
print("Clients:", train_frame["client_hash_id"].nunique())
print("Decline base rate:", round(float(y_cls.mean()), 4))
print("\nFold-level results:")
display(fold_results)
print("\nBefore/after summary:")
display(summary)
print("\nReceipt written:", receipt_path)


ASSIGNMENT 7 — HONEST SPLIT AUDIT
Pages: 1800
Clients: 15
Decline base rate: 0.7511

Fold-level results:


,split,fold,validation_pages,validation_clients,client_overlap,decline_base_rate,classification_roc_auc,regression_rmse,ranking_precision_at_50
0,random_page_cv,1,360,15,15,0.758333,0.745442,0.941418,1.00
1,random_page_cv,2,360,15,15,0.736111,0.743635,0.851227,0.94
2,random_page_cv,3,360,15,15,0.722222,0.730692,0.819649,0.94
3,random_page_cv,4,360,15,15,0.800000,0.679977,0.851611,0.92
4,random_page_cv,5,360,15,15,0.738889,0.723644,0.629250,0.94
5,grouped_client_cv,1,360,3,0,0.936111,0.697200,0.463367,1.00
6,grouped_client_cv,2,360,3,0,0.694444,0.528873,0.784814,0.70
7,grouped_client_cv,3,360,3,0,0.686111,0.689441,0.843647,0.86
8,grouped_client_cv,4,360,3,0,0.775000,0.808399,0.601016,1.00
9,grouped_client_cv,5,360,3,0,0.663889,0.601214,1.338529,0.80



Before/after summary:


,split,mean_client_overlap,mean_decline_base_rate,classification_roc_auc,classification_roc_auc_sd,regression_rmse,regression_rmse_sd,ranking_precision_at_50,ranking_precision_at_50_sd
0,grouped_client_cv,0.0,0.751111,0.665025,0.105826,0.806275,0.333493,0.872,0.130077
1,random_page_cv,15.0,0.751111,0.724678,0.026580,0.818631,0.115209,0.948,0.030332



Receipt written: ../outputs/assignment7_split_audit.json


## 3. Leakage audit

I audited the final five features against four leakage routes: **label-derived information**, **future-window overlap**, **decision-derived/product signals**, and **entity identifiers**. I also audit population selection separately because the modelling cohort requires sufficient April outcome observability.

The intended legal timeline is:

`March 1–31 features → prediction cutoff at March 31 → April 1–30 outcome`

The five active features are `aggregate_ctr`, `median_position`, `position_slope_per_day`, `position_iqr`, and `content_age_days`. IDs, usable-day counts, exposure strata, April values, and target fields are context/label fields rather than model inputs.

To verify that the audit harness can actually detect leakage, I deliberately add `future_impression_change`—the continuous quantity from which `future_decline` is derived—as an illegal feature and rerun grouped CV. That intentionally leaky result is diagnostic only and is never retained as a model result.

### Population-selection limitation

The final cohort requires at least 20 usable GSC days in April so that the future outcome can be measured reliably. This means row eligibility depends partly on information from the outcome window. It does **not** put April values into the five model features, but it conditions the evaluated population on future observability. Therefore my claims apply to March-eligible pages that also have sufficiently observed April outcomes; I do not generalize the measured performance to every March page without qualification.

### Real failure examples

I also refit the frozen Assignment-6 models on the 15 development clients and inspect concrete mistakes on the six held-out clients. The examples are shown without client or content identifiers. They are used to understand failure modes, not to retune the model.


In [ ]:
# SECTION 3 — leakage attack + real held-out failure examples.
# Relies on the exact modeling_frame reconstructed in Section 2.

from sklearn.model_selection import GroupKFold, cross_val_predict

# ---------- 3A. Static feature-role and timeline audit ----------
feature_audit = pd.DataFrame([
    {"feature": "aggregate_ctr", "source": "March clicks / March impressions", "available_by": "2026-03-31", "label_derived": False, "future_overlap": False, "decision_derived": False, "identifier": False},
    {"feature": "median_position", "source": "March daily GSC positions", "available_by": "2026-03-31", "label_derived": False, "future_overlap": False, "decision_derived": False, "identifier": False},
    {"feature": "position_slope_per_day", "source": "March daily GSC positions over time", "available_by": "2026-03-31", "label_derived": False, "future_overlap": False, "decision_derived": False, "identifier": False},
    {"feature": "position_iqr", "source": "March daily GSC positions", "available_by": "2026-03-31", "label_derived": False, "future_overlap": False, "decision_derived": False, "identifier": False},
    {"feature": "content_age_days", "source": "creation date measured at 2026-03-31", "available_by": "2026-03-31", "label_derived": False, "future_overlap": False, "decision_derived": False, "identifier": False},
])
feature_audit["verdict"] = np.where(
    feature_audit[["label_derived", "future_overlap", "decision_derived", "identifier"]].any(axis=1),
    "REVIEW",
    "PASS",
)

forbidden_fields = {
    "client_hash_id", "content_hash_id", "future_impression_change", "future_decline",
    "march_avg_impressions_per_day", "april_avg_impressions_per_day",
    "march_usable_days", "april_usable_days", "exposure_tier"
}
active_forbidden = sorted(set(FINAL_FEATURES).intersection(forbidden_fields))

assert len(FINAL_FEATURES) == 5
assert not active_forbidden
assert (feature_audit["verdict"] == "PASS").all()

# ---------- 3B. Outcome-window population-selection audit ----------
march20 = march_cov[march_cov["march_usable_days"] >= 20].copy()
coverage_pair = march20.merge(
    april_cov[["client_hash_id", "content_hash_id", "april_usable_days"]],
    on=["client_hash_id", "content_hash_id"],
    how="left",
)
coverage_pair["april_usable_days"] = coverage_pair["april_usable_days"].fillna(0).astype(int)
april_observable = coverage_pair[coverage_pair["april_usable_days"] >= 20].copy()

population_audit = {
    "march_feature_eligible_pages": int(len(march20)),
    "march_feature_eligible_clients": int(march20["client_hash_id"].nunique()),
    "april_observable_pages": int(len(april_observable)),
    "april_observable_clients": int(april_observable["client_hash_id"].nunique()),
    "march_pages_excluded_for_insufficient_april_observability": int(len(march20) - len(april_observable)),
    "pct_march_pages_retained_for_observable_outcome": float(100.0 * len(april_observable) / len(march20)),
}

# ---------- 3C. Deliberate leakage injection: prove the harness reacts ----------
# Compare mean fold-level AUC to mean fold-level AUC, matching Assignment 6 exactly.
grouped_cv = GroupKFold(n_splits=5)
grouped_splits = list(grouped_cv.split(X, y_cls, groups=groups))

X_leaky = X.copy()
# ILLEGAL diagnostic feature: the exact continuous quantity whose sign defines future_decline.
X_leaky["ILLEGAL_future_impression_change"] = y_reg.to_numpy()

legal_fold_auc = []
leaky_fold_auc = []

for fit_idx, valid_idx in grouped_splits:
    legal_model = clone(cls_model).fit(X.iloc[fit_idx], y_cls.iloc[fit_idx])
    legal_prob = legal_model.predict_proba(X.iloc[valid_idx])[:, 1]
    legal_fold_auc.append(
        float(roc_auc_score(y_cls.iloc[valid_idx], legal_prob))
    )

    leaky_model = clone(cls_model).fit(X_leaky.iloc[fit_idx], y_cls.iloc[fit_idx])
    leaky_prob = leaky_model.predict_proba(X_leaky.iloc[valid_idx])[:, 1]
    leaky_fold_auc.append(
        float(roc_auc_score(y_cls.iloc[valid_idx], leaky_prob))
    )

legal_auc = float(np.mean(legal_fold_auc))
leaky_auc = float(np.mean(leaky_fold_auc))
leakage_jump = leaky_auc - legal_auc

assert np.isclose(legal_auc, 0.6650253727996942)
assert leaky_auc > 0.98, "Deliberate leakage did not create near-perfect discrimination; inspect the harness."

# ---------- 3D. Real held-out failure examples, with identifiers removed ----------
test_clients = set(split_manifest["test_clients"])
test_frame = (
    modeling_frame[modeling_frame["client_hash_id"].isin(test_clients)]
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)
assert len(test_frame) == 720
assert set(train_frame["client_hash_id"]).isdisjoint(set(test_frame["client_hash_id"]))

X_test_audit = test_frame[FINAL_FEATURES].copy()
y_test_cls = test_frame["future_decline"].astype(int).copy()
y_test_reg = test_frame["future_impression_change"].astype(float).copy()

held_cls = clone(cls_model).fit(X, y_cls)
held_reg = clone(reg_model).fit(X, y_reg)
held_prob = held_cls.predict_proba(X_test_audit)[:, 1]
held_pred_cls = (held_prob >= 0.5).astype(int)
held_pred_reg = held_reg.predict(X_test_audit)

errors = test_frame[FINAL_FEATURES + ["future_decline", "future_impression_change"]].copy()
errors["p_decline"] = held_prob
errors["predicted_class"] = held_pred_cls
errors["predicted_future_change"] = held_pred_reg
errors["classification_error"] = np.select(
    [
        (errors["future_decline"] == 1) & (errors["predicted_class"] == 0),
        (errors["future_decline"] == 0) & (errors["predicted_class"] == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)
errors["regression_absolute_error"] = (
    errors["future_impression_change"] - errors["predicted_future_change"]
).abs()

fn_examples = (
    errors[errors["classification_error"] == "false_negative"]
    .sort_values("p_decline", ascending=True).head(2).copy()
)
fp_examples = (
    errors[errors["classification_error"] == "false_positive"]
    .sort_values("p_decline", ascending=False).head(2).copy()
)
reg_examples = errors.sort_values("regression_absolute_error", ascending=False).head(2).copy()

failure_examples = []
for label, frame in [("confident_false_negative", fn_examples), ("confident_false_positive", fp_examples)]:
    for _, row in frame.iterrows():
        failure_examples.append({
            "task": "classification",
            "failure_type": label,
            "actual_decline": int(row["future_decline"]),
            "predicted_decline_probability": float(row["p_decline"]),
            "realized_future_change": float(row["future_impression_change"]),
        })
for _, row in reg_examples.iterrows():
    failure_examples.append({
        "task": "regression",
        "failure_type": "largest_absolute_error",
        "actual_future_change": float(row["future_impression_change"]),
        "predicted_future_change": float(row["predicted_future_change"]),
        "absolute_error": float(row["regression_absolute_error"]),
    })
failure_examples_df = pd.DataFrame(failure_examples)

# Ranking held-out mistakes using the frozen Assignment-6 blend.
severity = np.maximum(0.0, -held_pred_reg)
severity_norm = np.clip(severity / SEVERITY_SCALE, 0.0, 1.0)
held_rank_score = np.power(np.clip(held_prob, 1e-9, 1.0), RANK_GAMMA) * (
    1.0 + RANK_LAMBDA * severity_norm
)
rank_errors = pd.DataFrame({
    "future_decline": y_test_cls.to_numpy(),
    "future_impression_change": y_test_reg.to_numpy(),
    "ranking_score": held_rank_score,
}).sort_values("ranking_score", ascending=False).reset_index(drop=True)
rank_errors["rank"] = np.arange(1, len(rank_errors) + 1)
top50_false_picks = rank_errors[(rank_errors["rank"] <= 50) & (rank_errors["future_decline"] == 0)].head(2)
missed_declines = rank_errors[(rank_errors["rank"] > 50) & (rank_errors["future_decline"] == 1)].sort_values("future_impression_change").head(2)

ranking_failure_examples = pd.concat([
    top50_false_picks.assign(failure_type="top50_false_pick"),
    missed_declines.assign(failure_type="large_decline_missed_outside_top50"),
], ignore_index=True)[["failure_type", "rank", "ranking_score", "future_impression_change"]]

leakage_receipt = {
    "active_features": FINAL_FEATURES,
    "forbidden_feature_overlap": active_forbidden,
    "feature_audit": feature_audit.to_dict(orient="records"),
    "population_selection": population_audit,
    "deliberate_leakage_test": {
        "legal_grouped_cv_roc_auc": legal_auc,
        "illegal_target_derived_feature_grouped_cv_roc_auc": leaky_auc,
        "roc_auc_jump": leakage_jump,
    },
    "heldout_error_counts": {
        "false_negatives": int(((errors["future_decline"] == 1) & (errors["predicted_class"] == 0)).sum()),
        "false_positives": int(((errors["future_decline"] == 0) & (errors["predicted_class"] == 1)).sum()),
        "top50_false_picks": int(((rank_errors["rank"] <= 50) & (rank_errors["future_decline"] == 0)).sum()),
        "declines_missed_outside_top50": int(((rank_errors["rank"] > 50) & (rank_errors["future_decline"] == 1)).sum()),
    },
}
leakage_receipt_path = output_dir / "assignment7_leakage_audit.json"
with open(leakage_receipt_path, "w", encoding="utf-8") as fh:
    json.dump(leakage_receipt, fh, indent=2)

print("FINAL FEATURE LEAKAGE AUDIT")
display(feature_audit)
print("Forbidden fields present in model features:", active_forbidden)

print("\nPOPULATION-SELECTION AUDIT")
display(pd.DataFrame([population_audit]))

print("\nDELIBERATE LEAKAGE TEST")
display(pd.DataFrame([{
    "legal_grouped_cv_roc_auc": legal_auc,
    "with_illegal_target_derived_feature": leaky_auc,
    "roc_auc_jump": leakage_jump,
}]))

print("\nREAL HELD-OUT CLASSIFICATION / REGRESSION FAILURE EXAMPLES")
display(failure_examples_df)

print("\nREAL HELD-OUT RANKING FAILURE EXAMPLES")
display(ranking_failure_examples)

print("\nLeakage/error receipt written:", leakage_receipt_path)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.